# Next-item prediction with the trained RecGPT

Load the **trained** RecGPT, feed a user's history, and generate the next
item's Semantic ID — decoded back into a name. Then measure ranking quality
on a **temporal split** (validation = 2nd-to-last item, test = last item).

*Kernel: `nanoTIGER (mps)`.*


In [1]:
# --- setup + device check -------------------------------------------------
import sys; sys.path.append('..')
import json, re
import numpy as np, torch
from config import cfg, DATA, OUT
from common import get_device, seed_everything
from tokenizer import build_tokenizer
from checkpoints import load_recgpt
from eval import beam_search_items, recall_ndcg
from demo_pipeline import ensure_trained

print('mps:', torch.backends.mps.is_available())
seed_everything(cfg.seed)
device = get_device()

mps: True
[device] using mps


In [2]:
ensure_trained()

[demo] using existing embeddings at /Users/gballoccu/Desktop/SemanticIDs/notebooks/../data/item_emb.npy
[demo] RQ-VAE already trained
[demo] RecGPT already trained


{'items': PosixPath('/Users/gballoccu/Desktop/SemanticIDs/notebooks/../data/items.jsonl'),
 'sequences': PosixPath('/Users/gballoccu/Desktop/SemanticIDs/notebooks/../data/sequences.json'),
 'semantic_ids': PosixPath('/Users/gballoccu/Desktop/SemanticIDs/notebooks/../data/semantic_ids.json'),
 'rqvae': PosixPath('/Users/gballoccu/Desktop/SemanticIDs/notebooks/../out/rqvae.pt'),
 'recgpt': PosixPath('/Users/gballoccu/Desktop/SemanticIDs/notebooks/../out/recgpt.pt')}

In [3]:
# --- load the trained model + tokenizer + data ---------------------------
import re
def short_name(text):
    m = re.search(r'<item_name>\s*(.*?)\s*</item_name>', text)
    return (m.group(1) if m else text)[:34]

items = [json.loads(l) for l in open(DATA / 'items.jsonl')]
items.sort(key=lambda r: r['item'])
names = [short_name(r['text']) for r in items]
cats  = [r.get('category') for r in items]

sem = json.load(open(DATA / 'semantic_ids.json'))
tok = build_tokenizer(sem['ids'])
sequences = json.load(open(DATA / 'sequences.json'))['train']
gpt, gptcfg = load_recgpt(OUT / 'recgpt.pt', device)
print('RecGPT vocab =', gptcfg.vocab_size, '| users =', len(sequences))

RecGPT vocab = 390 | users = 9000


**Generate & decode.** Beam-search the next item's Semantic ID, read it back as a name (test split).


In [4]:
# --- predict the next item, as names --------------------------------------
rng = np.random.default_rng(0)
for u in rng.choice(len(sequences), 4, replace=False):
    seq = sequences[int(u)]
    history, truth = seq[:-1], seq[-1]               # test split
    preds = beam_search_items(gpt, tok, history, device, cfg.beam_size, max(cfg.eval_ks))
    print(f'\nUser {u}  (domain: {cats[history[0]]})')
    print('  recent history:', [names[i] for i in history[-4:]])
    print('  top-{} predicted next items:'.format(max(cfg.eval_ks)))
    for i in preds:
        print(f'      {names[i]:34s} [{cats[i]}]')
    print('  actual next item:', names[truth], '->', 'HIT' if truth in preds else 'miss')


User 2428  (domain: Musical_Instruments)
  recent history: ['Yamaha Condenser Microphone 2024', 'Yamaha Overdrive Pedal', 'Yamaha Digital Piano', 'Yamaha Digital Piano']
  top-10 predicted next items:
      Yamaha Condenser Microphone Mini   [Musical_Instruments]
      Yamaha Instrument Cable Mini       [Musical_Instruments]
      Yamaha Dreadnought Guitar Lite     [Musical_Instruments]
      Yamaha Synthesizer 2024            [Musical_Instruments]
      Yamaha Snare Drum Mk II            [Musical_Instruments]
      Yamaha Condenser Microphone Mk II  [Musical_Instruments]
      Yamaha Digital Piano Pro           [Musical_Instruments]
      Yamaha Condenser Microphone XL     [Musical_Instruments]
      Yamaha Dreadnought Guitar          [Musical_Instruments]
      Yamaha Hi-Hat Cymbals              [Musical_Instruments]
  actual next item: Yamaha Synthesizer Pro -> miss

User 4599  (domain: Musical_Instruments)
  recent history: ['Yamaha Dreadnought Guitar XL', 'Yamaha Nylon Guitar 202

**Ranking quality on the temporal split.** Recall@K / NDCG@K for validation and test, vs a popularity baseline.


In [5]:
# --- top-K metrics on a sample (val + test) -------------------------------
from collections import Counter
ks = cfg.eval_ks
pop = Counter(i for s in sequences for i in s[:-2])   # train portion only
pop_rank = [i for i, _ in pop.most_common()]
sample = [int(u) for u in rng.choice(len(sequences), 250, replace=False)
          if len(sequences[int(u)]) >= 3]

def evaluate(which):
    R = {k: [0.0, 0.0] for k in ks}; P = {k: [0.0, 0.0] for k in ks}
    for u in sample:
        s = sequences[u]
        hist, tgt = (s[:-2], s[-2]) if which == 'val' else (s[:-1], s[-1])
        ranked = beam_search_items(gpt, tok, hist, device, cfg.beam_size, max(ks))
        unseen = [i for i in pop_rank if i not in set(hist)]
        for k in ks:
            r, g = recall_ndcg(ranked, tgt, k);  R[k][0] += r; R[k][1] += g
            pr, pg = recall_ndcg(unseen, tgt, k); P[k][0] += pr; P[k][1] += pg
    return R, P

n = len(sample)
print(f'{"":<11}{"val R@k":>9}{"test R@k":>10}{"test N@k":>10}{"pop R@k":>9}')
for which in ['x']:
    Rv, _ = evaluate('val'); Rt, Pt = evaluate('test')
    for k in ks:
        print(f'k={k:<9}{Rv[k][0]/n:>9.3f}{Rt[k][0]/n:>10.3f}{Rt[k][1]/n:>10.3f}{Pt[k][0]/n:>9.3f}')

             val R@k  test R@k  test N@k  pop R@k
k=5            0.088     0.088     0.044    0.000
k=10           0.140     0.184     0.075    0.000
